# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
%pip -q install transformers sentencepiece torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

| # | Task | Model family | Why |
|---|---|---|---|
| 1 | Sentiment classification | **Encoder-only** (e.g. BERT) | We only need to understand the text and output a label, no text generation. |
| 2 | Machine translation | **Encoder–decoder** (e.g. T5) | The encoder reads the sentence in one language and the decoder writes it in the other. |
| 3 | Open-ended text generation | **Decoder-only** (e.g. GPT) | It generates text one token at a time based on the previous tokens. |
| 4 | Named Entity Recognition | **Encoder-only** (e.g. BERT) | It's classifying each token (person, place, ...), so it needs to understand the full sentence from both sides. |
| 5 | Text summarization | **Encoder–decoder** (e.g. BART, T5) | It reads the whole input text and then generates a new shorter text. |

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [6]:
sentences = ["The movie was excellent.",
             "The lecture was difficult to understand.",
             "I enjoyed learning about transformers."]

results = classifier(sentences)

for sentence, res in zip(sentences, results):
    print(f"{sentence}\n  -> label: {res['label']}, score: {res['score']:.4f}\n")

The movie was excellent.
  -> label: POSITIVE, score: 0.9999

The lecture was difficult to understand.
  -> label: NEGATIVE, score: 0.9997

I enjoyed learning about transformers.
  -> label: POSITIVE, score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [7]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers understand, "
    "interpret and generate human language. It is used in many applications such as chatbots, machine "
    "translation, spam detection and sentiment analysis. In recent years, transformer models like BERT and GPT "
    "have improved the performance of NLP systems a lot, because they are trained on huge amounts of text and "
    "can understand the context of words in a sentence."
)

prompt = "Summarize the following text in one sentence: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=60)

print("Original length:", len(paragraph.split()), "words")
summary = tokenizer.decode(output[0], skip_special_tokens=True)
print("Summary:", summary)
print("Summary length:", len(summary.split()), "words")

Original length: 71 words
Summary: A natural language processing field is a field of artificial intelligence that helps computers understand, interpret and generate human language.
Summary length: 20 words


The summary went from 71 words to 20 words. It mostly took the first sentence of the paragraph and didn't mention the applications or transformers. I also tried just "Summarize: " but the output was worse ("Learn about natural language processing..."), so asking for one sentence worked better. This is because flan-t5-small is a very small model, a bigger model like flan-t5-base or BART would give a better summary.


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:**

A decoder-only model is trained to predict the next token based on all the tokens before it, and this is exactly what a chatbot does, it writes the reply one word at a time. In a chat, the whole conversation (previous messages + the new question) can be given to the model as one long prompt, and the model just continues it with the answer. It doesn't need a separate encoder because the input and the output are treated as one sequence. Also these models are trained on huge amounts of text, so they can generate long and fluent answers on almost any topic, which is what we need for open-ended conversation.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [8]:
# with a fixed seed
for i in range(3):
    set_seed(42)
    out = generator("Natural language processing is", max_new_tokens=20, do_sample=True)
    print(i + 1, ":", out[0]["generated_text"])

1 : Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words


2 : Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words


3 : Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words


In [9]:
# without set_seed
for i in range(3):
    out = generator("Natural language processing is", max_new_tokens=20, do_sample=True)
    print(i + 1, ":", out[0]["generated_text"])

1 : Natural language processing is an important part of the overall English-speaking language. It is a complex learning process which can arise


2 : Natural language processing is a crucial part of many languages. Many languages are thought of as a function of the word order.


3 : Natural language processing is now the foundation of the modern language processing process. It’s a necessary part of the human


**Answer:**

With `do_sample=True` the model doesn't always pick the word with the highest probability. It randomly picks the next token based on the probabilities, so each run can choose a different word, and after one different word the rest of the sentence changes too. When I used `set_seed(42)` before each run, all 3 outputs were exactly the same because the random number generator starts from the same point every time. Without it, the 3 outputs were different. In the first example in the lab (`do_sample=False`) the model uses greedy decoding, so it always gives the same output.